# Session 1

## Introduction to Biological Networks and Knowledge Graphs

### General

This session is about representing biomedical knowledge as a **graph**, and about
the specific ways that representation misleads us if we are not careful.

We build a breast-cancer-focused knowledge graph and then interrogate it: starting
from clinical ICD-10 codes, recovering information the graph appears not to have,
and finally connecting it to the molecular data used in Session 2.

The graph is deliberately small — 881 nodes — so that every result can be checked
by eye. Everything here scales to graphs of millions of nodes without changing.

The data are:

| File | Description |
|------|-------------|
| `data/kg_nodes.csv` | Node table — `id`, `type` (gene / disease / icd10), `name`, `extra` |
| `data/kg_edges.csv` | Edge table — `source`, `target`, `type`, `weight`, `evidence` |
| `data/kg_evidence.csv` | The same gene–disease edges split by *kind* of evidence |
| `data/icd10_map.csv` | MONDO disease id → ICD-10 code and label |
| `data/coexpr_expression.csv.gz` | 500 patients × 737 genes of TCGA-BRCA expression |

Built from [Open Targets](https://platform.opentargets.org/) (CC0 1.0) and the
[MONDO Disease Ontology](https://mondo.monarchinitiative.org/) (CC BY 4.0) by
`data-prep/build_kg_data.py`. That script downloads ~1.1 GB and cuts it down;
we do not need to run it.

Genes are keyed by **Ensembl gene ID**, the same identifier space as the TCGA-BRCA
transcriptomics matrix in Session 2, so the two datasets join natively. The
expression file is that matrix, cut down to the genes that are already nodes in
the graph — which is what lets us build an inferred *and* a curated network over
exactly the same genes.

### Part 1 — What is a Network, and What is a Knowledge Graph?

#### 1.1 Graphs

A graph is a set of **nodes** and a set of **edges** between them. Degree, paths and
components all follow from those two sets. This part establishes the vocabulary,
using a diagram and a four-node toy example, and shows the one operation that makes
graphs worth the trouble: composing edges into a path produces an answer nobody
stored.

#### 1.2 Inferred versus curated

Two networks can look identical and mean opposite things.

An **inferred** network computes its edges from measurements. Correlate every gene
against every other gene, keep what clears a threshold. The edges are a statistical
claim about *our data*, and the network can contain relationships nobody has
described before.

A **curated knowledge graph** reads its edges out of a database of recorded facts.
It can only contain what somebody already knew.

The failure modes are opposite, and neither is fixable by better code. In an
inferred network a spurious edge is usually a confounder. In a knowledge graph a
*missing* edge usually means nobody has looked yet.

#### 1.3 What makes it a knowledge graph

Typed nodes, typed edges, and provenance. Ours has three node types (gene, disease,
icd10) and three edge types (`associated_with`, `is_a`, `maps_to`), with an
association score and evidence count on every gene–disease edge. That is the
"specialised" in *a knowledge graph is a specialised class of network*.

#### 1.4 Building one of each

We then build both networks over the same 737 genes and compare them. `BRCA1`
co-expresses with the proliferation programme and is curated against hereditary
breast–ovarian cancer; `TP53` is a 27-disease hub in the curated graph and
essentially invisible to co-expression. Same genes, same identifiers, two
different pictures — and the reasons they differ are the point.

### Part 2 — Building a Knowledge Graph with NetworkX

#### 2.1 Construction

A knowledge graph is a loop over a node table and an edge table. We write it out
explicitly before switching to the helper, so that nothing is magic.

One subtlety gets its own treatment: `is_a` is genuinely directional, but we work
undirected because degree and components behave better that way. The direction is
preserved as an edge attribute — without it, climbing the ontology in Part 3 walks
downwards half the time and returns plausible wrong answers rather than errors.

#### 2.2 Shape

We read the graph's summary statistics and ask what each one is really telling us.
The density is low, the clustering coefficient is near zero (the graph is close to
bipartite), and the hubs are all diseases.

That last one is a **construction artefact**: the build kept the top 30 genes per
disease, so diseases were always going to be the hubs. The habit this part is
trying to build is asking, every time, whether a graph property came from the data
or from how the data was cut.

#### 2.3 Annotation sparsity

The most important idea in the session. A missing edge has three possible causes —
the relationship does not exist, nobody has studied it, or it was recorded against
a different term — and the graph cannot tell us which.

We meet a concrete case: basal-like breast carcinoma, one of the most studied
breast cancer subtypes, has **zero** gene associations here, because the evidence
was filed against the near-synonymous term triple-negative breast carcinoma.

### Part 3 — Practical: Querying the Knowledge Graph

#### 3.1 Starting from ICD-10

[ICD-10](https://icd.who.int/browse10/2019/en) is the vocabulary hospitals
actually use. Only 19 of our 90 diseases carry a code — and the gap is not spread
evenly: **3 of 63** breast subtypes against **16 of 27** other diseases.

The reason is not a bad data source. ICD-10 subdivides breast cancer
**anatomically** — by quadrant — so there is no code for "triple-negative" or
"luminal A", and there cannot be one. Morphology lives in a separate
classification entirely. This is a **granularity mismatch** between the clinical
and molecular vocabularies, and no alternative database fixes it.

Climbing the `is_a` hierarchy recovers a code for 82 of 90 — with the caveat that
an inherited code describes the ancestor, not the subtype we started from.

#### 3.2 Which diseases share genes?

Projecting the bipartite gene–disease graph onto diseases alone turns "how similar
are these two diseases?" into a structural question. Breast and ovarian cancer
share BRCA1, BRCA2 and BRIP1 — hereditary breast-ovarian cancer syndrome,
recovered from graph structure without being told the two diseases were related.

The same result also surfaces tubulins, shared not because of biology but because
both cancers are treated with taxanes. The overall association score cannot
distinguish the two — but the **evidence types** can. Filter to genetic and somatic
evidence and every tubulin disappears.

Community detection on that causal graph then recovers the clinical taxonomy —
cancers, autoimmune diseases, metabolic diseases — without being told any of it exists.

#### 3.3 Connecting to the omics data

Finally, the bridge to Session 2. The TCGA matrix uses versioned Ensembl IDs
(`ENSG00000012048.23`) and Open Targets does not, so a naive join matches
**nothing at all** — not fewer things, nothing. An empty join is the *safe*
failure; a partial one is what reaches publication.

All five PAM50 subtypes predicted in Session 2 are disease nodes in this graph, so
a subtype prediction becomes an entry point rather than a label.

#### 3.4 Both networks at once

The closing section puts the co-expression network from Part 1 back alongside the
curated graph. The genes with causal evidence for breast cancer partly corroborate
each other in expression (`BRCA2`, `BRIP1`, `BARD1` — the repair complex). But
their strongest co-expression partners include the *same* chemotherapy targets
Section B just filtered out, now arriving for a different reason. Two sources
agreeing is only evidence when their errors are independent.

### Where this leads

Session 3 builds LLM agents — tools, MCP and skills — and Session 4 turns them on
graphs like this one, queried from multi-omics profiles. Everything in this
session — the coverage gaps, the inherited codes, the
technically-correct-but-misleading edges, the candidate list nothing in the data
can sort — is what those agents have to get right, and what we need to check them
against.